# Phase 2 — Streaming Pipeline

**Project:** Aegis — Cost-Aware Real-Time Fraud Intelligence System
**Author:** Kashish Lalwani
**Course:** AMS 561, Spring 2026

## What this notebook does
Wraps the cleaned Phase 1 Delta table in a Spark Structured Streaming job that simulates real-time
transaction arrival, engineers two domain-informed features in-stream (`is_high_risk_type` and
`balance_diff`), and writes the enriched stream to a downstream Delta sink for the Phase 3 model.

## Inputs
- `dbfs:/Volumes/workspace/default/raw_data/clean` (Phase 1 output, 49,962 rows × 11 columns)

## Outputs
- `dbfs:/Volumes/workspace/default/raw_data/streaming_output` (49,962 rows × 13 columns)
- `dbfs:/Volumes/workspace/default/raw_data/checkpoint` (streaming progress checkpoint)

## Why streaming for a static dataset?
A real payments processor sees transactions arrive continuously, not in nightly batches. Spark
Structured Streaming lets us write the same logic that would run in production micro-batch
processing, fault-tolerant checkpointing, append-mode writes against a static file. Setting
`maxFilesPerTrigger=1` makes Spark consume one Delta file per micro-batch, which simulates
transactions arriving over time.

Step 1 - Verifying our clean data is readable

In [0]:
df = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/raw_data/clean")
print(f"Row count: {df.count()}")
df.printSchema()

Row count: 49962
root
 |-- step: double (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: double (nullable = true)



Step 2 — Defining the schema explicitly

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("step", DoubleType(), True),
    StructField("type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("nameOrig", StringType(), True),
    StructField("oldbalanceOrg", DoubleType(), True),
    StructField("newbalanceOrig", DoubleType(), True),
    StructField("nameDest", StringType(), True),
    StructField("oldbalanceDest", DoubleType(), True),
    StructField("newbalanceDest", DoubleType(), True),
    StructField("isFraud", IntegerType(), True),
    StructField("isFlaggedFraud", DoubleType(), True)
])

print("Schema defined successfully.")

Schema defined successfully.


Step 3 — Reading the data as a stream

In [0]:
streaming_df = (
    spark.readStream
    .format("delta")
    .option("maxFilesPerTrigger", 1)
    .load("dbfs:/Volumes/workspace/default/raw_data/clean")
)

print("Is streaming:", streaming_df.isStreaming)

Is streaming: True


Step 4 — Adding feature columns

Two domain-informed features are added directly inside the stream rather than as a separate batch step. This reflects how features would be computed in production at the time the transaction arrives, not retroactively.

- **`is_high_risk_type`** — binary flag set to 1 for `TRANSFER` and `CASH_OUT` transactions, the only two types where fraud occurs in this dataset (verified in Phase 1). This single feature captures most of the model's signal.
- **`balance_diff`** — `oldbalanceOrg − newbalanceOrig`, the drop in the sender's balance. A behavioral indicator of fund drainage that is informative even when the transaction type alone is ambiguous.

Both features feed directly into the Random Forest model in Phase 3.

In [0]:
from pyspark.sql.functions import col, when

featured_df = streaming_df.withColumn(
    "is_high_risk_type",
    when(col("type").isin("TRANSFER", "CASH_OUT"), 1).otherwise(0)
).withColumn(
    "balance_diff",
    col("oldbalanceOrg") - col("newbalanceOrig")
)

print("Features added. Columns:", featured_df.columns)

Features added. Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'is_high_risk_type', 'balance_diff']


Step 5 — Writing the stream to a sink

In [0]:
# Clear stale streaming state before re-running the pipeline
# This is needed when the source Delta table has been overwritten (e.g., re-running Phase 1)
# Streaming checkpoints assume an append-only source; an overwrite invalidates them.

import shutil

paths_to_clear = [
    "/Volumes/workspace/default/raw_data/checkpoint",
    "/Volumes/workspace/default/raw_data/streaming_output",
]

for path in paths_to_clear:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"Cleared: {path}")
    except Exception as e:
        print(f"Could not clear {path} (may not exist yet): {e}")

print("\nReady to re-run the streaming pipeline.")

Cleared: /Volumes/workspace/default/raw_data/checkpoint
Cleared: /Volumes/workspace/default/raw_data/streaming_output

Ready to re-run the streaming pipeline.


In [0]:
from pyspark.sql.streaming import DataStreamWriter

query = (
    featured_df.writeStream
    .format("delta")
    .option("checkpointLocation", "dbfs:/Volumes/workspace/default/raw_data/checkpoint")
    .outputMode("append")
    .trigger(availableNow=True)
    .start("dbfs:/Volumes/workspace/default/raw_data/streaming_output")
)

query.awaitTermination()

Step 6 — Verifying the output

In [0]:
output_df = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/raw_data/streaming_output")

print(f"Total rows written: {output_df.count()}")
print(f"Columns: {output_df.columns}")
output_df.select("type", "amount", "is_high_risk_type", "balance_diff", "isFraud").show(5)

Total rows written: 49962
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'is_high_risk_type', 'balance_diff']
+--------+---------+-----------------+------------+-------+
|    type|   amount|is_high_risk_type|balance_diff|isFraud|
+--------+---------+-----------------+------------+-------+
|CASH_OUT|118326.41|                1|   118326.41|      0|
| PAYMENT| 17068.83|                0|    17068.83|      0|
|CASH_OUT|369264.57|                1|    124268.0|      0|
| PAYMENT| 10039.08|                0|         0.0|      0|
| PAYMENT|  7639.89|                0|         0.0|      0|
+--------+---------+-----------------+------------+-------+
only showing top 5 rows


## Summary

- **Read** 49,962 rows from the Phase 1 Delta table as a Spark Structured Stream
- **Engineered** two features in-stream: `is_high_risk_type` (binary) and `balance_diff` (double)
- **Wrote** 49,962 enriched rows (13 columns) to the Phase 3 input table in append mode
- **Pipeline runtime:** ~17.5 seconds, 2 tasks completed, 2.53 MB written
- **Checkpoint** persisted for restart-safe re-runs

**Saved to:** `dbfs:/Volumes/workspace/default/raw_data/streaming_output` (Delta Lake)

**Next:** Phase 3 (`03_fraud_model.ipynb`) reads this enriched table, assembles the seven model features into a vector column, and trains a class-weighted Random Forest classifier.